# Project Canary — Harvest-Recovery Forecast

**Purpose:** reproduce the complete forecasting workflow in a form the capstone team can run and defend.

**Business question:** using only records known on a review date, what last-recorded harvest-recovery proxy should we expect for this building, compared with the 95% goal?

This notebook does not calculate the independent rules-based risk score and does not prove that any input causes recovery to change.

## 1. Define Y, X, and the unit of analysis

- **Y target:** population on the building's last recorded daily date ÷ beginning population.
- **Important:** this is the agreed capstone recovery proxy, not a verified harvest-event label.
- **One outcome:** one building in one completed cycle.
- **One training snapshot:** that building's facts known at a selected age. To avoid overweighting long cycles, training retains Days 7, 14, 21, 28, plus the last eligible pre-outcome snapshot.
- **Candidate X inputs:** production age; current survival; recent mortality; weight gap and measurement freshness; and temperature/humidity deviations from approved age bands. Feed is withheld until its recorded unit is confirmed. The compact set excludes building identity, raw inventory size, and algebraic duplicates.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "canary").exists():
    ROOT = ROOT.parent
DATA_PATH = ROOT / "data" / "FARM HARVEST DATA.xlsx"
MODEL_READY_DIR = ROOT / "outputs" / "model_ready"

from canary import load_workbook

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 140)
dataset = load_workbook(DATA_PATH)
print(f"Source: {dataset.source_name}")
print(f"Canonical building-day rows: {len(dataset.daily):,}")
print(f"Recorded building-cycles: {len(dataset.cycles):,}")
print(f"Blocking data-quality checks passed: {dataset.quality.passed}")
print(f"Non-blocking warnings: {len(dataset.quality.warnings)}")

Source: FARM HARVEST DATA.xlsx
Canonical building-day rows: 1,666
Recorded building-cycles: 34
Blocking data-quality checks passed: True
Non-blocking warnings: 3


In [2]:
from canary import build_modeling_snapshots, build_recovery_training_snapshots, train_outcome_model

daily_snapshots = build_modeling_snapshots(dataset, "recovery")
training_snapshots = build_recovery_training_snapshots(dataset)
coverage = pd.DataFrame({
    "Measure": ["Complete cycles", "Distinct building outcomes", "All leakage-safe daily snapshots", "Balanced decision snapshots"],
    "Count": [training_snapshots["cycle_id"].nunique(), training_snapshots[["cycle_id", "building_id"]].drop_duplicates().shape[0], len(daily_snapshots), len(training_snapshots)],
})
coverage

,Measure,Count
0,Complete cycles,5
1,Distinct building outcomes,25
2,All leakage-safe daily snapshots,1122
3,Balanced decision snapshots,122


In [3]:
exported = pd.read_csv(MODEL_READY_DIR / "recovery_training.csv")
assert len(exported) == len(training_snapshots) == 122
expected_keys = set(zip(training_snapshots.cycle_id.astype(str), training_snapshots.building_id, training_snapshots.as_of_date.astype(str)))
exported_keys = set(zip(exported.cycle_id.astype(str), exported.building_id, exported.as_of_date.astype(str)))
assert exported_keys == expected_keys
print("Export reconciliation passed: the CSV contains the exact 122 balanced recovery snapshots.")

Export reconciliation passed: the CSV contains the exact 122 balanced recovery snapshots.


## 2. Preprocessing and validation

1. Convert the workbook to one canonical building-day row; zone rows are aggregated before modeling.
2. Construct every snapshot with records dated on or before its review date; later records are excluded.
3. Give each building-cycle equal total weight despite repeated checkpoints.
4. Use **nested leave-one-complete-cycle-out cross-validation**: the outer loop tests a completely unseen cycle; the inner loop tunes only within the remaining cycles. Imputation and scaling stay inside those folds.
5. Compare exactly five declared candidates primarily on **cycle-macro MAE**. A learned recovery model must improve the historical mean by at least 10% and keep positive whole-cycle R² before replacing the baseline for the continuous estimate.

No random row split is used because rows from the same flock history are related and would leak information across train and test sets.

In [4]:
result = train_outcome_model(dataset, "recovery")
manifest = result.manifest
print("Operational continuous estimator:", manifest["selected_model"])
print("Best learned challenger:", manifest["research_champion"])
print("Champion gates:", manifest["champion_gates"])
print("Model version:", manifest["model_version"])
print("Selected X inputs:")
for feature in manifest["feature_columns"]:
    print(" -", feature)

Operational continuous estimator: linear_regression
Best learned challenger: linear_regression
Champion gates: {'baseline': 'historical_mean', 'baseline_improvement_pct': 14.531176046433437, 'requires_at_least_10pct_mae_improvement': True, 'requires_positive_r2': True, 'regression_gate_passed': True, 'requires_better_than_majority_target_side_accuracy': False, 'requires_recall_for_both_target_sides': True, 'target_classification_gate_passed': False, 'operational_fallback_applied': False}
Model version: recovery-1.0.0
Selected X inputs:
 - cycle_day
 - percentage_alive
 - mortality_recent_3d_per_1000
 - mortality_trend_delta_per_1000
 - weight_gap_pct
 - weight_staleness_days
 - temperature_deviation_from_band_c
 - humidity_deviation_from_band_pp
 - environment_out_of_band_days_7d
 - environment_staleness_days


## 3. Candidate comparison

In [5]:
comparison = pd.DataFrame([
    {
        "Candidate": entry["model"],
        "Available": entry["available"],
        "Role": "Operational" if entry["model"] == manifest["selected_model"] else "Best learned challenger" if entry["model"] == manifest["research_champion"] else "Compared",
        "MAE (points)": manifest["metrics"].get(entry["model"], {}).get("mae", np.nan) * 100,
        "Cycle-macro MAE (points)": manifest["metrics"].get(entry["model"], {}).get("cycle_macro_mae", np.nan) * 100,
        "RMSE (points)": manifest["metrics"].get(entry["model"], {}).get("rmse", np.nan) * 100,
        "R²": manifest["metrics"].get(entry["model"], {}).get("r2", np.nan),
        "Target-side accuracy": manifest["metrics"].get(entry["model"], {}).get("target_side_accuracy", np.nan),
    }
    for entry in manifest["candidate_registry"]
]).sort_values("Cycle-macro MAE (points)")
comparison.round({"MAE (points)": 2, "Cycle-macro MAE (points)": 2, "RMSE (points)": 2, "R²": 3, "Bias (points)": 2, "Target-side accuracy": 3})

,Candidate,Available,Role,MAE (points),Cycle-macro MAE (points),RMSE (points),R²,Target-side accuracy
1,linear_regression,True,Operational,1.37,1.48,1.84,0.189,0.828
2,ridge_core,True,Compared,1.55,1.59,1.97,0.070,0.836
3,gradient_boosting,True,Compared,1.57,1.66,2.08,-0.041,0.844
0,historical_mean,True,Compared,1.66,1.73,2.17,-0.132,0.844
4,xgboost,False,Compared,NaN,NaN,NaN,NaN,NaN


In [6]:
cycle_performance = pd.DataFrame.from_dict(manifest["selected_metrics"]["cycle"], orient="index")
cycle_performance.index.name = "Held-out cycle"
cycle_performance.assign(
    mae_points=cycle_performance.mae * 100,
    rmse_points=cycle_performance.rmse * 100,
    bias_points=cycle_performance.bias * 100,
)[["rows", "mae_points", "rmse_points", "bias_points"]].round(2)

,rows,mae_points,rmse_points,bias_points
Held-out cycle,,,,
2025-2,12,2.59,2.64,-2.59
2025-3,25,0.68,0.89,0.51
2025-4,25,0.94,1.15,-0.08
2025-5,30,1.47,1.71,-0.66
2026-1,30,1.71,2.50,1.64


In [7]:
selected = manifest["selected_metrics"]
print(f"Selected held-out MAE: {selected['mae']*100:.2f} percentage points")
print(f"Selected held-out RMSE: {selected['rmse']*100:.2f} percentage points")
print(f"80% empirical error half-width: ±{selected['uncertainty_half_width_80']*100:.2f} points")
print(f"Target-side accuracy: {selected['target_side_accuracy']:.1%}")
print(f"Majority baseline accuracy: {selected['majority_side_accuracy']:.1%}")

Selected held-out MAE: 1.37 percentage points
Selected held-out RMSE: 1.84 percentage points
80% empirical error half-width: ±2.10 points
Target-side accuracy: 82.8%
Majority baseline accuracy: 84.4%


**Interpretation:** the operational learned estimator clears the continuous MAE/R² gate, but its target-side accuracy does not beat the majority baseline. Present it as a prototype continuous estimate with uncertainty—not as a proven classifier of 95% target attainment.

## 4. What the selected model relies on

In [8]:
importance = pd.DataFrame(manifest["held_out_permutation_importance"])
importance.head(10).rename(columns={
    "feature": "Input",
    "mean_mae_increase": "Held-out MAE increase",
    "relative_importance_pct": "Relative held-out reliance (%)",
}).round(4)

,Input,Held-out MAE increase,Relative held-out reliance (%)
0,humidity_deviation_from_band_pp,0.0096,28.2878
1,percentage_alive,0.0087,25.7263
2,cycle_day,0.0040,11.7009
3,mortality_recent_3d_per_1000,0.0035,10.3264
4,mortality_trend_delta_per_1000,0.0021,6.3263
5,temperature_deviation_from_band_c,0.0017,5.1564
6,environment_staleness_days,0.0016,4.5832
7,environment_out_of_band_days_7d,0.0015,4.4579
8,weight_gap_pct,0.0009,2.7048
9,weight_staleness_days,0.0002,0.7298


These are out-of-fold permutation importances from complete unseen cycles. They show predictive reliance and are **associations, not causal effects**.

## 5. Day 14 held-out proof and one complete example

In [9]:
def cycle_bootstrap_mae(frame, error_column, repeats=5000, seed=42):
    # Bootstrap whole cycles, never individual rows, to preserve grouped evidence.
    rng = np.random.default_rng(seed)
    grouped = {cycle: group for cycle, group in frame.groupby("cycle_id")}
    cycles = np.array(list(grouped))
    estimates = []
    for _ in range(repeats):
        selected = rng.choice(cycles, size=len(cycles), replace=True)
        errors = np.concatenate([grouped[cycle][error_column].to_numpy(float) for cycle in selected])
        estimates.append(np.mean(np.abs(errors)))
    return np.quantile(estimates, [0.025, 0.975])

In [10]:
day14 = pd.DataFrame(manifest["day14_backtest"])
day14["error_points"] = day14["error"] * 100
day14["absolute_error_points"] = day14["absolute_error"] * 100
ci = cycle_bootstrap_mae(day14, "error_points")
metrics = manifest["day14_backtest_metrics"]
print(f"Day 14 building outcomes: {metrics['building_cycles']}")
print(f"Day 14 MAE: {metrics['mae']*100:.2f} points")
print(f"Cycle-bootstrap 95% interval for Day 14 MAE: {ci[0]:.2f} to {ci[1]:.2f} points")
example = day14.iloc[0]
print("\nExample")
print(f"Cycle/building: {example.cycle_id} / {example.building_id}")
print(f"Day 14 held-out projection: {example.predicted:.1%}")
print(f"Last-recorded actual proxy: {example.actual:.1%}")
print(f"Error = projected - actual: {example.error_points:+.2f} percentage points")
day14.head(8)[["cycle_id", "building_id", "predicted", "actual", "error_points"]]

Day 14 building outcomes: 25
Day 14 MAE: 1.65 points
Cycle-bootstrap 95% interval for Day 14 MAE: 1.17 to 2.05 points

Example
Cycle/building: 2025-2 / Tags 1
Day 14 held-out projection: 92.6%
Last-recorded actual proxy: 94.3%
Error = projected - actual: -1.64 percentage points


,cycle_id,building_id,predicted,actual,error_points
0,2025-2,Tags 1,0.926413,0.942794,-1.638086
1,2025-2,Tags 2,0.928538,0.949706,-2.116796
2,2025-2,Tags 3,0.930301,0.955588,-2.528685
3,2025-3,Lags 1,0.918055,0.900036,1.801967
4,2025-3,Lags 2,0.905590,0.907909,-0.231880
5,2025-3,Tags 1,0.946624,0.938072,0.855248
6,2025-3,Tags 2,0.938771,0.931068,0.770301
7,2025-3,Tags 3,0.951713,0.944998,0.671557


## 6. Why SMOTE or oversampling is not used

- The outcome is continuous regression, while standard SMOTE is designed for classification.
- The scarce item is the number of independent building-cycle outcomes—not the number of spreadsheet rows. Synthetic rows do not create new farms or cycles.
- Interpolating flock records could create biologically implausible combinations and falsely narrow validation error.
- Oversampling before grouped validation could leak the held-out cycle.

**Safer small-data strategy used here:** simple regularized candidates, complete-cycle holdouts, balanced checkpoints, empirical uncertainty, cycle-level bootstrap intervals, and transparent limitations. The strongest improvement is collecting more standardized completed cycles with verified harvest events.

## 7. Defense takeaway

Canary's recovery output is a **nested whole-cycle-validated estimate of the agreed last-recorded recovery proxy**. Its held-out error is roughly 1–2 percentage points, but it is not yet strong at recognizing the small number of outcomes at or above 95%. Use it to size likely gaps and guide attention, not to claim certainty.